# 1. The data

This notebook reads the raw recordings of every dataset and writes the files that the other notebooks use, in
`cache/`. It is long because every dataset has its own file format. It is slow (a few hours the first time): each
part is skipped when its file is already there.

## Where to get the datasets

Download and unzip by hand into `data/`. Nothing here is ours to share again.

| Folder in `data/` | Dataset | What we use |
|---|---|---|
| `cluster/` | [Cluster haptic texture dataset](https://doi.org/10.6084/m9.figshare.29438288) | sound, vibration, `texture_list.xlsx` |
| `lmt108/` | [LMT 108 surface materials](https://zeus.lkn.ei.tum.de/downloads/texture/) | `SoundScans/Movement`, `AccelScansComponents/Movement` |
| `lmt69/` | [LMT texture database 1.4](https://zeus.lkn.ei.tum.de/downloads/texture/) | the zip file, not unzipped |
| `hatt/raw/`, `hatt/raw_friction/` | [Penn Haptic Texture Toolkit](https://repository.upenn.edu/handle/20.500.14332/30085) | the two archives of recordings |
| `tum_building_materials/recordings/` | [TUM building materials](https://mediatum.ub.tum.de/1474014) | one `.npz` per recording, with the fields of the original files |
| `sens3/` | [SENS3](https://www.sens3.net/surfaces) | `Sliding_Friction-roughness`, `Tapping_Hardness` |
| `devillard/` | [Devillard 2025](https://doi.org/10.6084/m9.figshare.26965267) | `datah5.zip`, not unzipped |
| `greatest_hits/` | [Greatest Hits](https://andrewowens.com/vis/hits.html) | `*_denoised.wav`, `*_times.txt` |
| `own_recordings/` | our phones | `pilot/wav`, `colleague_g/original` |

BEATs (code and the checkpoint `BEATs_iter3_plus_AS2M.pt`) goes in `models/beats/`, from
[the BEATs repository](https://github.com/microsoft/unilm/tree/master/beats).

In [1]:
import io
import json
import re
import subprocess
import sys
import zipfile
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchaudio
from scipy import signal
from scipy.io import wavfile

DATA = Path("data")
Path("cache").mkdir(exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

## The families

`families.csv` is our own table: for each surface of each dataset, wood / metal / textile / polymer, or nothing
when the surface fits none of them (stone, glass, paper...).

In [2]:
families = pd.read_csv("families.csv", dtype={"surface": str}).dropna(subset="family")


def family_of(dataset):
    part = families[families.dataset == dataset]
    return dict(zip(part.surface, part.family))


families.groupby("dataset").family.value_counts().unstack(fill_value=0)

family,metal,polymer,textile,wood
dataset,,,,
cluster,9,10,18,35
greatest_hits,1,1,2,1
hatt,5,13,22,3
lmt108,5,8,16,7
sens3,5,7,12,6
tum,3,3,1,2


## Small tools

In [3]:
def same_level(x):
    # every window gets the same loudness
    return x / (np.sqrt(np.mean(x**2, axis=-1, keepdims=True)) + 1e-12)


def main_axis(xyz):
    # three axes of an accelerometer -> one signal, along the direction that moves the most
    centered = xyz - xyz.mean(axis=0)
    return centered @ np.linalg.svd(centered, full_matrices=False)[2][0]


def sound_windows(samples, rate):
    # 0.5 s windows at 16 kHz
    samples = signal.resample_poly(samples, 160, rate // 100)
    return samples[: len(samples) // 8000 * 8000].reshape(-1, 8000)

## Cluster and LMT108: sound and vibration of the same scrape

For each recording we keep the part where the probe touches the surface, then compute:

- one power spectrum of the sound and one of the vibration (for `signals.ipynb`);
- small spectrograms of 0.5 s (for the CNN of `known_setup.ipynb`);
- the raw 0.5 s windows (for the pretrained models).

In [4]:
CLUSTER = DATA / "cluster/cluster_haptic_texture_dataset/sensor_data"
LMT108 = DATA / "lmt108/LMT_108_SurfaceMaterials_Database"


def list_two_setups():
    rows = []
    for path in sorted((CLUSTER / "audio").rglob("*.wav")):
        surface, direction, velocity, force, repeat = path.stem.split("_")
        rows.append({"dataset": "cluster", "surface": surface, "direction": int(direction), "velocity": int(velocity),
                     "force": int(force), "repeat": int(repeat), "audio": path,
                     "accel": [CLUSTER / "accel" / surface / f"{path.stem}.csv"]})
    for path in sorted((LMT108 / "SoundScans/Movement").rglob("*.wav")):
        surface, trial = path.stem.split("_Sound_Movement_")
        folder = LMT108 / "AccelScansComponents/Movement" / path.parent.name
        rows.append({"dataset": "lmt108", "surface": surface, "audio": path,
                     "accel": [folder / f"{surface}_Movement_{axis}_{trial}.txt" for axis in "XYZ"]})
    table = pd.DataFrame(rows).merge(families[["dataset", "surface", "family"]], on=["dataset", "surface"])
    return table.reset_index(drop=True)


def read_two_setups(row):
    # sound at 44.1 kHz, vibration at 8 kHz (3 axes), and the seconds where the contact starts and stops
    audio = wavfile.read(row["audio"])[1].astype(float)
    if audio.ndim > 1:
        audio = audio[:, 0]
    if row["dataset"] == "lmt108":
        accel = signal.resample_poly(np.column_stack([np.loadtxt(p) for p in row["accel"]]), 4, 5, axis=0)
        return audio, accel, 0.1, len(audio) / 44100 - 0.1
    values = np.loadtxt(row["accel"][0], delimiter=",", skiprows=1)
    times = np.arange(values[0, 0], values[-1, 0], 1 / 8000)
    accel = np.column_stack([np.interp(times, values[:, 0], values[:, axis]) for axis in [1, 2, 3]])
    # Cluster: the probe lands (a shock on the accelerometer), slides, then leaves (the sound stops)
    beginning = accel[:3600]
    touchdown = np.argmax(np.abs(beginning - np.median(beginning, axis=0)).sum(axis=1)) / 8000
    frames = audio[: len(audio) // 2205 * 2205].reshape(-1, 2205)
    envelope = np.sqrt(np.mean(frames**2, axis=1))
    middle = envelope[len(envelope) // 4 : 3 * len(envelope) // 4]
    end = np.flatnonzero(envelope > 0.5 * np.median(middle))[-1] * 0.05
    return audio, accel, touchdown + 0.2, end - 0.15


mel = torchaudio.transforms.MelSpectrogram(sample_rate=16000, n_fft=512, hop_length=128, n_mels=64, f_min=50, f_max=8000)
spectrogram = torchaudio.transforms.Spectrogram(n_fft=512, hop_length=64, power=2)


def log_shape(power):
    log_power = torch.log(power + 1e-10)
    return (log_power - log_power.mean()).numpy()


def band_level(x, rate, band):
    filtered = signal.sosfiltfilt(signal.butter(4, band, "bandpass", fs=rate, output="sos"), x, axis=0)
    return np.log10(np.sqrt(np.mean(filtered**2)))


def process(row):
    torch.set_num_threads(1)
    audio, accel, start, stop = read_two_setups(row)
    audio = audio[int(start * 44100) : int(stop * 44100)]
    accel = accel[int(start * 8000) : int(stop * 8000)]
    accel = accel - accel.mean(axis=0)
    out = {
        "sound spectrum": signal.welch(audio, 44100, nperseg=8192)[1][:2049],
        "vibration spectrum": signal.welch(accel, 8000, nperseg=1600, axis=0)[1].sum(axis=1),
        "audio_log_rms": band_level(audio, 44100, (100, 8000)),
        "accel_log_rms": band_level(accel, 8000, (10, 600)),
    }
    audio_16k = signal.resample_poly(audio, 160, 441)
    n = min(8, len(audio_16k) // 8000, len(accel) // 4000)
    sound = np.stack([audio_16k[k * 8000 : (k + 1) * 8000] for k in range(n)])
    vibration = [accel[k * 4000 : (k + 1) * 4000] for k in range(n)]
    out["sound windows"] = same_level(sound).astype(np.float16)
    out["vibration windows"] = same_level(np.stack([main_axis(v) for v in vibration])).astype(np.float16)
    out["sound images"] = np.stack([log_shape(mel(torch.tensor(s, dtype=torch.float32))) for s in sound]).astype(np.float16)
    out["vibration images"] = np.stack(
        [log_shape(spectrogram(torch.tensor(v.T, dtype=torch.float32)).sum(dim=0)[1:39]) for v in vibration]
    ).astype(np.float16)
    return out


if not Path("cache/spectra.npz").exists():
    two_setups = list_two_setups()
    with Pool() as pool:
        done = pool.map(process, [row for _, row in two_setups.iterrows()], chunksize=8)
    info = two_setups.drop(columns=["audio", "accel"])
    info["audio_log_rms"] = [d["audio_log_rms"] for d in done]
    info["accel_log_rms"] = [d["accel_log_rms"] for d in done]
    info.to_csv("cache/two_setups.csv", index=False)
    np.savez("cache/spectra.npz", sound=np.array([d["sound spectrum"] for d in done], dtype=np.float32),
             vibration=np.array([d["vibration spectrum"] for d in done], dtype=np.float32),
             **{c: info[c].to_numpy() for c in info.columns})
    owner = np.concatenate([np.full(len(d["sound windows"]), k) for k, d in enumerate(done)])
    np.savez("cache/spectrograms.npz", audio=np.concatenate([d["sound images"] for d in done]),
             accel=np.concatenate([d["vibration images"] for d in done]), recording=owner)
    np.savez("cache/windows_two_setups.npz", sound=np.concatenate([d["sound windows"] for d in done]),
             vibration=np.concatenate([d["vibration windows"] for d in done]), recording=owner)

## SENS3: a bare finger

Each session is a long recording of a finger sliding back and forth. We cut it in pieces of 4 s and keep the
pieces where the finger really presses and moves.

In [5]:
SENS3 = DATA / "sens3"


def list_sens3():
    rows = []
    for surface, family in family_of("sens3").items():
        for session in sorted((SENS3 / f"Material{surface}/Sliding_Friction-roughness").iterdir()):
            force = -np.load(session / f"Material_sensor_{surface}.npy", mmap_mode="r")[:, 5]
            position = pd.read_csv(session / f"Material_IR_pos_{surface}.csv", header=None)
            seconds = len(wavfile.read(session / f"Material_{surface}.wav", mmap=True)[1]) / 44100
            duration = min(len(force) / 10000, seconds, position[0].iloc[-1])
            for k in range(int(duration // 4)):
                pressing = force[k * 40000 : (k + 1) * 40000] > 0.1
                speed = np.median(position[1][(position[0] >= 4 * k) & (position[0] < 4 * k + 4)])
                if pressing.mean() >= 0.95 and speed >= 20:
                    rows.append({"dataset": "sens3", "surface": surface, "family": family, "velocity": speed,
                                 "session": str(session), "start": 4 * k})
    return pd.DataFrame(rows)


def process_sens3(row):
    session, surface, start = Path(row["session"]), row["surface"], row["start"]
    audio = wavfile.read(session / f"Material_{surface}.wav", mmap=True)[1][start * 44100 : (start + 4) * 44100]
    sensor = np.load(session / f"Material_sensor_{surface}.npy", mmap_mode="r")[start * 10000 : (start + 4) * 10000, :3]
    audio_16k = signal.resample_poly(audio.astype(float), 160, 441)
    accel = signal.resample_poly(np.asarray(sensor, dtype=float), 4, 5, axis=0)
    sound = np.stack([audio_16k[k * 8000 : (k + 1) * 8000] for k in range(8)])
    vibration = np.stack([main_axis(accel[k * 4000 : (k + 1) * 4000]) for k in range(8)])
    return same_level(sound).astype(np.float16), same_level(vibration).astype(np.float16)


if not Path("cache/windows_sens3.npz").exists():
    sens3 = list_sens3()
    with Pool() as pool:
        done = pool.map(process_sens3, [row for _, row in sens3.iterrows()], chunksize=8)
    sens3.to_csv("cache/sens3.csv", index=False)
    np.savez("cache/windows_sens3.npz", sound=np.concatenate([d[0] for d in done]),
             vibration=np.concatenate([d[1] for d in done]), recording=np.repeat(np.arange(len(done)), 8))

## BEATs

A pretrained audio model. We never train it: it only turns a 0.5 s window into a vector.
The vibration is at 8 kHz, BEATs wants 16 kHz, so we resample it first.

In [6]:
sys.path.insert(0, "models/beats")
from BEATs import BEATs, BEATsConfig

checkpoint = torch.load("models/beats/BEATs_iter3_plus_AS2M.pt", map_location="cpu", weights_only=True)
beats = BEATs(BEATsConfig(checkpoint["cfg"]))
beats.load_state_dict(checkpoint["model"])
beats = beats.to(device).eval()


def beats_vectors(windows, rate=8000):
    # one vector per 0.5 s window
    out = []
    with torch.no_grad():
        for start in range(0, len(windows), 128):
            x = torch.tensor(np.asarray(windows[start : start + 128], dtype=np.float32))
            if rate != 16000:
                x = torchaudio.functional.resample(x, rate, 16000)
            out.append(beats.extract_features(x.to(device) * 0.1)[0].mean(dim=1).cpu().numpy())
    return np.concatenate(out)

## Vibration vectors of the three first setups and of HaTT

For `few_labels.ipynb`. We keep at most 40 recordings per surface of Cluster and 80 of SENS3, drawn at random.
HaTT has two files of 10 s per texture: we skip the first 0.5 s and cut two recordings of 4 s in each.

In [7]:
def read_hatt(path):
    # the HaTT files are XML with one <value> per sample
    blocks = {}
    for name, body in re.findall(r"<([A-Za-z_]+)>\s*(<value>.*?)</\1>", path.read_text(), flags=re.S):
        blocks[name] = np.array(re.findall(r"<value>([^<]+)</value>", body), dtype=float)
    return blocks


HATT_FOLDERS = [DATA / "hatt/raw", DATA / "hatt/raw_friction"]

if not Path("cache/vibration_vectors.npz").exists():
    two = pd.read_csv("cache/two_setups.csv", dtype={"surface": str}).assign(file="two_setups")
    sens3 = pd.read_csv("cache/sens3.csv", dtype={"surface": str}).assign(file="sens3")
    two["number"], sens3["number"] = np.arange(len(two)), np.arange(len(sens3))
    everything = pd.concat([two, sens3], ignore_index=True)[["dataset", "surface", "family", "number", "file"]]
    limit = {"cluster": 40, "lmt108": None, "sens3": 80}
    rng = np.random.default_rng(0)
    selected = []
    for (dataset, _), group in everything.groupby(["dataset", "surface"], sort=True):
        if limit[dataset] and len(group) > limit[dataset]:
            group = group.iloc[np.sort(rng.choice(len(group), limit[dataset], replace=False))]
        selected.append(group)
    selected = pd.concat(selected, ignore_index=True)
    selected.to_csv("cache/selected_recordings.csv", index=False)

    vectors = np.zeros((len(selected), 768), np.float32)
    for file, group in selected.groupby("file"):
        saved = np.load(f"cache/windows_{file}.npz")
        keep = np.isin(saved["recording"], group.number)
        per_window = beats_vectors(saved["vibration"][keep])
        vectors[group.index] = pd.DataFrame(per_window).groupby(saved["recording"][keep]).mean().loc[group.number].to_numpy()

    rows, windows = [], []
    for folder in HATT_FOLDERS:
        for texture in sorted(folder.iterdir()):
            if texture.name in family_of("hatt"):
                accel = read_hatt(texture / f"Accel_{texture.name}.xml")
                xyz = signal.resample_poly(np.stack([accel["Accel_x"], accel["Accel_y"], accel["Accel_z"]], axis=1), 4, 5, axis=0)
                for start in range(4000, len(xyz) - 32000 + 1, 32000):
                    windows += [same_level(main_axis(xyz[start + k * 4000 : start + (k + 1) * 4000])) for k in range(8)]
                    rows.append({"dataset": "hatt", "surface": texture.name, "family": family_of("hatt")[texture.name]})
    hatt_vectors = beats_vectors(np.array(windows)).reshape(len(rows), 8, -1).mean(axis=1)

    info = pd.concat([selected[["dataset", "surface", "family"]], pd.DataFrame(rows)], ignore_index=True)
    info["surface"] = info.dataset + ":" + info.surface
    np.savez("cache/vibration_vectors.npz", vectors=np.concatenate([vectors, hatt_vectors]),
             **{c: info[c].to_numpy() for c in info.columns})

## Sound windows of every dataset that has sound

For CLAP (`new_setup.ipynb`, `fine_tuning.ipynb`): 0.5 s at 16 kHz, same loudness, silent windows removed.

- LMT108, Cluster, SENS3: 12 recordings per surface, from the windows above.
- LMT69: ten people scrape each surface (inside the zip).
- TUM: the sound with the steel tooltip, wood / metal / polymer only.
- Greatest Hits: the "scratch" events of the videos, at most 300 per family.
- Our phone: 5 objects.

In [8]:
LMT69_ZIP = DATA / "lmt69/LMT_TextureDB_1.4.zip"
TUM = DATA / "tum_building_materials/recordings"
lmt_family = {k.replace("G4Beech_0", "G4Beech"): v for k, v in family_of("lmt108").items()}
tum_family = family_of("tum")


def lmt69_sounds():
    archive = zipfile.ZipFile(LMT69_ZIP)
    folder = "___LMT_TextureDB___/Testing/SoundScans/"
    for name in sorted(n for n in archive.namelist() if n.startswith(folder + "G") and n.endswith(".wav")):
        surface, person = Path(name).stem.split("_Sound_")
        rate, samples = wavfile.read(io.BytesIO(archive.read(name)))
        yield surface, person, samples.astype(float).reshape(len(samples), -1)[:, 0], rate


def tum_sounds():
    for path in sorted(TUM.glob("*.npz")):
        recording = np.load(path, allow_pickle=True)
        if "soundSteelTooltip" in recording.files:
            yield path.stem.rsplit("_I", 1)[0], path.stem.rsplit("_I", 1)[1], recording["soundSteelTooltip"], 44100


def greatest_hits_scratches():
    material_family = family_of("greatest_hits")
    rows = []
    for times in sorted((DATA / "greatest_hits").glob("*_times.txt")):
        video = times.name.removesuffix("_times.txt")
        rate, samples = wavfile.read(DATA / f"greatest_hits/{video}_denoised.wav", mmap=True)
        events = pd.read_csv(times, sep="\\s+", header=None, names=["onset", "material", "action", "reaction"])
        next_onset = np.append(events.onset.to_numpy()[1:], len(samples) / rate)
        for onset, material, action, after in zip(events.onset, events.material, events.action, next_onset):
            if action == "scratch" and material in material_family and after - onset >= 0.5:
                rows.append({"surface": f"{video} {material}", "record": f"{video} {onset}", "onset": onset,
                             "family": material_family[material], "video": video})
    return pd.DataFrame(rows)


if not Path("cache/sound_windows.npz").exists():
    rows, windows = [], []

    def add(dataset, recording, surface, family, pieces):
        level = np.sqrt((pieces**2).mean(axis=1, keepdims=True))
        pieces = (pieces / (level + 1e-12))[level[:, 0] > 0]
        windows.append(pieces.astype(np.float16))
        rows.extend([{"dataset": dataset, "recording": f"{dataset} {recording}", "surface": surface, "family": family}] * len(pieces))

    selected = pd.read_csv("cache/selected_recordings.csv", dtype={"surface": str})
    saved = {f: np.load(f"cache/windows_{f}.npz") for f in ["two_setups", "sens3"]}

    def from_selected(dataset, name):
        for row in selected[selected.dataset == dataset].groupby("surface").head(12).itertuples():
            pieces = saved[row.file]["sound"][saved[row.file]["recording"] == row.number].astype(np.float32)
            add(name, f"{row.surface} {row.number}", row.surface, row.family, pieces)

    from_selected("lmt108", "LMT108")
    for surface, person, samples, rate in lmt69_sounds():
        if surface in lmt_family:
            add("LMT69", f"{surface} {person}", surface, lmt_family[surface], sound_windows(samples, rate))
    for surface, instance, samples, rate in tum_sounds():
        family = tum_family.get("_".join(surface.split("_")[:2]))
        if family in ["wood", "metal", "polymer"]:
            add("TUM", f"{surface}_I{instance}", surface, family, sound_windows(samples, rate))
    from_selected("sens3", "SENS3")
    from_selected("cluster", "Cluster")
    scratches = greatest_hits_scratches()
    scratches = pd.concat([g.sample(min(len(g), 300), random_state=0) for _, g in scratches.groupby("family")])
    for row in scratches.itertuples():
        rate, samples = wavfile.read(DATA / f"greatest_hits/{row.video}_denoised.wav", mmap=True)
        chunk = np.asarray(samples[int(row.onset * rate) : int(row.onset * rate) + rate // 2], dtype=float)
        if chunk.ndim > 1:
            chunk = chunk.mean(axis=1)
        if len(chunk) == rate // 2:
            add("Greatest Hits", row.record, row.surface, row.family, sound_windows(chunk, rate))
    pilot_family = {"metal": "metal", "bois": "wood", "textile": "textile", "plastique": "polymer"}
    for path in sorted((DATA / "own_recordings/pilot/wav").glob("*.wav")):
        parts = path.stem.split("_")
        if len(parts) >= 3 and parts[2] != "coups":
            rate, samples = wavfile.read(path)
            add("Phone pilot", path.stem, "_".join(parts[:2]), pilot_family[parts[0]], sound_windows(samples / 2**31, rate))

    table = pd.DataFrame(rows)
    np.savez("cache/sound_windows.npz", windows=np.concatenate(windows), **{c: table[c].to_numpy() for c in table.columns})

The colleague's phone, kept apart: these 28 recordings are only ever used as a test.

In [9]:
if not Path("cache/colleague_windows.npz").exists():
    word_family = {"jean": "textile", "metal": "metal", "plastic": "polymer", "bois": "wood"}
    rows, windows = [], []
    for path in sorted((DATA / "own_recordings/colleague_g/original").glob("G *.m4a")):
        command = ["ffmpeg", "-v", "error", "-i", str(path), "-f", "f32le", "-ac", "1", "-ar", "48000", "-"]
        samples = np.frombuffer(subprocess.run(command, capture_output=True, check=True).stdout, dtype=np.float32)
        pieces = sound_windows(samples.astype(np.float64), 48000)
        level = np.sqrt((pieces**2).mean(axis=1, keepdims=True))
        pieces = (pieces / (level + 1e-12))[level[:, 0] > 0]
        words = path.stem.lower().split()
        windows.append(pieces.astype(np.float16))
        rows += [{"recording": path.stem, "family": word_family[words[1]], "tool": words[2]}] * len(pieces)
    table = pd.DataFrame(rows)
    np.savez("cache/colleague_windows.npz", windows=np.concatenate(windows), **{c: table[c].to_numpy() for c in table.columns})

## Vibration of the steel tips, the fingers and the machine, at 3 kHz

For the locked test of `new_setup.ipynb` and for `friction.ipynb`. The TUM recordings are sampled at 3 kHz, so
every other dataset goes through 3 kHz too before BEATs.

One generator per dataset. Each gives (surface, name of the recording, signal, sample rate).

In [10]:
DEVILLARD_ZIP = DATA / "devillard/datah5.zip"


def lmt108_vibration():
    folder = LMT108 / "AccelScansComponents/Movement"
    for half in ["Training", "Testing"]:
        for x_file in sorted((folder / half).glob("*_Movement_X_*.txt")):
            surface, trial = x_file.stem.split("_Movement_X_")
            axes = [np.loadtxt(x_file.with_name(f"{surface}_Movement_{axis}_{trial}.txt")) for axis in "XYZ"]
            yield surface.replace("G4Beech_0", "G4Beech"), trial, np.column_stack(axes), 10000


def lmt69_vibration():
    # ten people (3 axes) and four passes of the operator (1 axis) per surface
    archive = zipfile.ZipFile(LMT69_ZIP)
    inside = "___LMT_TextureDB___/"
    for name in sorted(n[len(inside) :] for n in archive.namelist()):
        if name.startswith("Testing/AccelScansComponents/G") and name.endswith("_X.txt"):
            surface, person = Path(name).stem[:-2].split("_")
            axes = [np.loadtxt(archive.open(inside + name.replace("_X.txt", f"_{axis}.txt"))) for axis in "XYZ"]
            yield surface, person, np.column_stack(axes), 10000
        if name.startswith("Training/Accel/G") and "_query" in name:
            surface, query = Path(name).stem.split("_query")
            if int(query) < 4:
                yield surface, "operator " + query, np.loadtxt(archive.open(inside + name))[:, None], 10000


def cluster_vibration():
    for surface in sorted((CLUSTER / "accel").iterdir()):
        files = sorted(surface.glob("*.csv"))
        for path in files[:: max(1, len(files) // 12)][:12]:
            values = np.loadtxt(path, delimiter=",", skiprows=1)
            times = np.arange(values[0, 0], values[-1, 0], 1 / 8000)
            xyz = np.column_stack([np.interp(times, values[:, 0], values[:, axis]) for axis in [1, 2, 3]])
            yield surface.name, path.stem, xyz[4000:-4000], 8000


def tum_vibration():
    for path in sorted(TUM.glob("*.npz")):
        recording = np.load(path, allow_pickle=True)
        if "accelDFTSteelTooltip" in recording.files:
            yield path.stem.rsplit("_I", 1)[0], path.stem.rsplit("_I", 1)[1], recording["accelDFTSteelTooltip"][:, None], 3000


def hatt_vibration():
    for half, folder in zip(["first", "second"], HATT_FOLDERS):
        for texture in sorted(folder.iterdir()):
            accel = read_hatt(texture / f"Accel_{texture.name}.xml")
            yield texture.name, half, np.stack([accel["Accel_x"], accel["Accel_y"], accel["Accel_z"]], axis=1), 10000


def sens3_sessions():
    for material in sorted(SENS3.glob("Material*"), key=lambda p: int(p.name.removeprefix("Material"))):
        surface = material.name.removeprefix("Material")
        for session in sorted((material / "Sliding_Friction-roughness").iterdir()):
            yield surface, session, np.load(session / f"Material_sensor_{surface}.npy", mmap_mode="r")


def sens3_pieces(sensor):
    # starts of the 4 s pieces where the finger presses, at most 20 per session
    force = -np.asarray(sensor[:, 5])
    pressing = force > 0.5 * np.percentile(force, 95)
    starts = [s for s in range(0, len(sensor) - 40000, 40000) if pressing[s : s + 40000].mean() > 0.5]
    return starts[:: max(1, len(starts) // 20)][:20]


def sens3_vibration():
    for surface, session, sensor in sens3_sessions():
        for start in sens3_pieces(sensor):
            yield surface, f"{session.name} {start}", np.asarray(sensor[start : start + 40000, :3], dtype=float), 10000


def devillard_trials():
    # sensors at 12.5 kHz: left microphone, right microphone, nail accelerometer, phalanx accelerometer
    import h5py

    archive = zipfile.ZipFile(DEVILLARD_ZIP)
    for name in sorted(n for n in archive.namelist() if n.endswith(".h5")):
        _, person, texture, file_name = name.split("/")
        with h5py.File(io.BytesIO(archive.read(name)), "r") as trial:
            yield texture, f"{person} {file_name[:-3]}", trial["kistler_data"][:], trial["ft_sensor_data"][:]


def devillard_vibration():
    for texture, trial, sensors, _ in devillard_trials():
        nail = signal.resample_poly(sensors[:, 2], 4, 5)
        starts = np.linspace(20000, len(nail) - 60000, 4).astype(int)
        if starts[0] >= 0 and starts[-1] > starts[0]:
            for start in starts:
                yield texture, f"{trial} {start}", nail[start : start + 40000, None], 10000


def vibration_vector(xyz, rate):
    # down to 3 kHz, back to 8 kHz, 0.5 s windows on the main axis, BEATs, mean
    if rate != 3000:
        xyz = signal.resample_poly(xyz, 3, rate // 1000, axis=0)
    xyz = signal.resample_poly(xyz, 8, 3, axis=0)
    windows = [same_level(main_axis(xyz[k * 4000 : (k + 1) * 4000])) for k in range(len(xyz) // 4000)]
    return beats_vectors(np.array(windows)).mean(axis=0) if windows else None


if not Path("cache/vibration_vectors_3khz.npz").exists():
    sources = {"HaTT": hatt_vibration, "LMT69": lmt69_vibration, "LMT108": lmt108_vibration, "TUM": tum_vibration,
               "SENS3": sens3_vibration, "Devillard": devillard_vibration, "Cluster": cluster_vibration}
    label = {"HaTT": family_of("hatt"), "LMT69": lmt_family, "LMT108": lmt_family, "SENS3": family_of("sens3"),
             "Cluster": family_of("cluster")}
    rows, vectors = [], []
    for dataset, make in sources.items():
        for surface, _, xyz, rate in make():
            vector = vibration_vector(xyz, rate)
            if vector is not None:
                family = tum_family.get("_".join(surface.split("_")[:2]), "") if dataset == "TUM" else label.get(dataset, {}).get(surface, "")
                rows.append({"dataset": dataset, "surface": surface, "family": family})
                vectors.append(vector)
    table = pd.DataFrame(rows)
    np.savez("cache/vibration_vectors_3khz.npz", vectors=np.array(vectors), **{c: table[c].to_numpy() for c in table.columns})

The sound of the same sessions, for the test "friction from sound" of `friction.ipynb`.

In [11]:
def sens3_sounds():
    for surface, session, sensor in sens3_sessions():
        rate, samples = wavfile.read(session / f"Material_{surface}.wav", mmap=True)
        for start in sens3_pieces(sensor):
            chunk = np.asarray(samples[int(start / 10000 * rate) : int(start / 10000 * rate) + 4 * rate], dtype=float)
            if len(chunk) == 4 * rate:
                yield surface, f"{session.name} {start}", chunk.reshape(len(chunk), -1)[:, 0], rate


def devillard_sounds():
    for texture, trial, sensors, _ in devillard_trials():
        starts = np.linspace(25000, len(sensors) - 75000, 4).astype(int)
        if starts[0] >= 0 and starts[-1] > starts[0]:
            for start in starts:
                yield texture, f"{trial} {start}", sensors[start : start + 50000, 0], 12500


if not Path("cache/sound_vectors_beats.npz").exists():
    rows, vectors = [], []
    for dataset, make in {"LMT69": lmt69_sounds, "SENS3": sens3_sounds, "TUM": tum_sounds, "Devillard": devillard_sounds}.items():
        for surface, _, samples, rate in make():
            windows = sound_windows(samples, rate)
            if len(windows):
                rows.append({"dataset": dataset, "surface": surface})
                vectors.append(beats_vectors(same_level(windows), rate=16000).mean(axis=0))
    table = pd.DataFrame(rows)
    np.savez("cache/sound_vectors_beats.npz", vectors=np.array(vectors), **{c: table[c].to_numpy() for c in table.columns})

## Taps

TUM and SENS3 also recorded the same surfaces being **tapped**: the vibration of the tool or of the finger, and the
sound. For `few_labels.ipynb` we keep the 4 strongest taps of each recording (0.5 s each, starting just before the
peak), then one BEATs vector per recording.

LMT108 has taps too, but only in the full archive, which we did not download.

In [12]:
def strongest_taps(x, length, n=4):
    x = np.asarray(x, np.float64)
    if len(x) < length:
        return [np.pad(x, (0, length - len(x)))] if len(x) > length // 10 else []
    envelope, found = np.abs(x - x.mean()), []
    floor = np.median(envelope) + 1e-12
    for _ in range(n):
        peak = int(envelope.argmax())
        if envelope[peak] < 5 * floor:
            break
        start = max(0, peak - length // 10)
        if start + length <= len(x):
            found.append(x[start : start + length])
        envelope[max(0, peak - length // 2) : peak + length // 2] = 0
    return found


def tap_vector(pieces, rate):
    pieces = np.stack(pieces)
    pieces = same_level(pieces - pieces.mean(axis=1, keepdims=True))
    return beats_vectors(pieces, rate=rate).mean(axis=0)


if not Path("cache/tap_vectors.npz").exists():
    rows, vectors = [], []
    for path in sorted(TUM.glob("*.npz")):
        recording = np.load(path, allow_pickle=True)
        surface = path.stem.rsplit("_I", 1)[0]
        if "accelTapSteelTooltip" in recording.files:
            taps = strongest_taps(signal.resample_poly(np.asarray(recording["accelTapSteelTooltip"], float), 8, 3), 4000)
            if taps:
                rows.append(["TUM", surface, "tap vibration"]); vectors.append(tap_vector(taps, 8000))
        if "soundTapSteelTooltip" in recording.files:
            taps = strongest_taps(signal.resample_poly(np.asarray(recording["soundTapSteelTooltip"], float), 160, 441), 8000)
            if taps:
                rows.append(["TUM", surface, "tap sound"]); vectors.append(tap_vector(taps, 16000))
    for material in sorted(SENS3.glob("Material*"), key=lambda p: int(p.name.removeprefix("Material"))):
        surface = material.name.removeprefix("Material")
        for person in sorted((material / "Tapping_Hardness").iterdir()):
            xyz = signal.resample_poly(pd.read_csv(person / "accelerations.csv", header=None).to_numpy(float), 4, 5, axis=0)
            taps = strongest_taps(main_axis(xyz), 4000)
            if taps:
                rows.append(["SENS3", surface, "tap vibration"]); vectors.append(tap_vector(taps, 8000))
            rate, samples = wavfile.read(person / "audio.wav")
            samples = samples.astype(float).reshape(len(samples), -1)[:, 0]
            taps = strongest_taps(signal.resample_poly(samples, 16000, rate), 8000)
            if taps:
                rows.append(["SENS3", surface, "tap sound"]); vectors.append(tap_vector(taps, 16000))
    table = pd.DataFrame(rows, columns=["dataset", "surface", "signal"])
    np.savez("cache/tap_vectors.npz", vectors=np.array(vectors), **{c: table[c].to_numpy() for c in table.columns})

## Measured friction

Friction coefficient of a surface = median of (tangential force / normal force) while the tool or the finger
presses. LMT69 only gives a friction force by one operator, so for it we can only compare the order of the surfaces.
Cluster gives the coefficient of each texture in its list.

In [13]:
def coefficient(tangential, normal):
    pressing = np.abs(normal) > 0.5 * np.percentile(np.abs(normal), 95)
    return float(np.median(np.abs(tangential[pressing]) / np.abs(normal[pressing])))


if not Path("cache/measured_friction.csv").exists():
    rows = []
    for folder in HATT_FOLDERS:
        for texture in sorted(folder.iterdir()):
            force = read_hatt(texture / f"Force_{texture.name}.xml")
            rows.append(["HaTT", texture.name, coefficient(force["ForceTangential"], force["ForceNormal"])])
    archive = zipfile.ZipFile(LMT69_ZIP)
    folder = "___LMT_TextureDB___/Training/Friction/"
    for name in sorted(n for n in archive.namelist() if n.startswith(folder) and n.endswith("_continuosFric.txt")):
        rows.append(["LMT69", Path(name).stem.replace("_continuosFric", ""), float(np.median(np.loadtxt(archive.open(name))))])
    for path in sorted(TUM.glob("*.npz")):
        recording = np.load(path, allow_pickle=True)
        if "frictionForce" in recording.files and "normalForce" in recording.files:
            rows.append(["TUM", path.stem.rsplit("_I", 1)[0], coefficient(recording["frictionForce"], recording["normalForce"])])
    for surface, _, sensor in sens3_sessions():
        normal = np.clip(-np.asarray(sensor[:, 5]), 0, None)  # the finger only pushes down
        rows.append(["SENS3", surface, coefficient(np.hypot(np.asarray(sensor[:, 3]), np.asarray(sensor[:, 4])), normal)])
    # Devillard: the force sensor carries the texture plate, its resting force is given per texture
    resting = json.loads(zipfile.ZipFile(DEVILLARD_ZIP).read("datah5/texture_metadata.json"))
    for texture, _, _, force in devillard_trials():
        force = force[:, :3] - np.array(resting[texture]["force_offset"])
        rows.append(["Devillard", texture, coefficient(np.hypot(force[:, 0], force[:, 1]), force[:, 2])])
    from openpyxl import load_workbook

    sheet = list(load_workbook(DATA / "cluster/cluster_haptic_texture_dataset/texture_list.xlsx", read_only=True, data_only=True).active.values)
    cluster_list = pd.DataFrame(sheet[1:], columns=sheet[0])
    for number, value in zip(cluster_list["Texture_id"], pd.to_numeric(cluster_list["Dynamic friction coefficient"], errors="coerce")):
        rows.append(["Cluster", str(int(number)), value])

    friction = pd.DataFrame(rows, columns=["dataset", "surface", "friction"])
    friction = friction[np.isfinite(friction.friction) & (friction.friction > 0)]
    friction.groupby(["dataset", "surface"], as_index=False).friction.mean().to_csv("cache/measured_friction.csv", index=False)

## What is in `cache/` now

In [14]:
for path in sorted(Path("cache").glob("*")):
    print(f"{path.stat().st_size / 1e6:8.1f} MB  {path.name}")

    31.1 MB  clap_vectors.npy
     1.4 MB  clap_vectors_colleague.npy
    11.1 MB  colleague_windows.npz
    37.6 MB  frozen_beats_sound.npy
    37.6 MB  frozen_beats_vibration.npy
    25.1 MB  frozen_clap_sound.npy
    25.1 MB  frozen_clap_vibration.npy
     0.0 MB  measured_friction.csv
     0.5 MB  sens3.csv
    16.4 MB  sound_vectors_beats.npz
   243.4 MB  sound_windows.npz
   140.2 MB  spectra.npz
     4.7 MB  spectral_shapes.csv
   665.4 MB  spectrograms.npz
     2.9 MB  tap_vectors.npz
    19.1 MB  vibration_vectors.npz
    29.7 MB  vibration_vectors_3khz.npz
   956.7 MB  windows_sens3.npz
  1242.1 MB  windows_two_setups.npz
